# 02 · CRUD en MongoDB

Colección `events` (documentos originales). Driver: `pymongo`.

**Justificación:** el payload varía por tipo de evento → esquema flexible, consultas por campos anidados, agregaciones.

In [1]:
import sys, os
sys.path.insert(0, "../src")          # modulos del proyecto
import config
from datetime import datetime, timezone, date
from pprint import pprint

In [2]:
db = config.mongo_db()
col = db.events
print('BD:', db.name, '| documentos:', col.estimated_document_count())

BD: github_events | documentos: 283096


## C · Crear documento

In [3]:
demo_id = 'demo-' + datetime.now().strftime('%H%M%S')
doc = {'_id': demo_id, 'type': 'WatchEvent', 'actor': {'id': 1, 'login': 'estudiante_utec'},
       'repo': {'id': 1, 'name': 'utec/bigdata-demo'}, 'public': True,
       'created_at': datetime.now(timezone.utc), 'payload': {'action': 'started'}}
res = col.insert_one(doc)
print('insertado:', res.inserted_id)

insertado: demo-201732


## R · Consultar documentos

In [4]:
pprint(col.find_one({'_id': demo_id}))

{'_id': 'demo-201732',
 'actor': {'id': 1, 'login': 'estudiante_utec'},
 'created_at': datetime.datetime(2026, 11, 10, 20, 17, 32, 988000, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC')),
 'payload': {'action': 'started'},
 'public': True,
 'repo': {'id': 1, 'name': 'utec/bigdata-demo'},
 'type': 'WatchEvent'}


In [5]:
# consulta con filtro, proyeccion y orden
for d in col.find({'type': 'PushEvent'}, {'actor.login': 1, 'repo.name': 1, 'created_at': 1}).sort('created_at', -1).limit(5):
    print(d)

{'_id': '26629816244', 'actor': {'login': 'github-actions[bot]'}, 'created_at': datetime.datetime(2026, 11, 9, 7, 3, 41, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC')), 'repo': {'name': 'eliezeradler/twitter-bot'}}
{'_id': '26629816346', 'actor': {'login': 'github-actions[bot]'}, 'created_at': datetime.datetime(2026, 11, 9, 7, 3, 41, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC')), 'repo': {'name': 'piseytha-pixel/quarterly-sequence'}}
{'_id': '26629816369', 'actor': {'login': 'Bhambya'}, 'created_at': datetime.datetime(2026, 11, 9, 7, 3, 41, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC')), 'repo': {'name': 'Bhambya/homelab'}}
{'_id': '26629816384', 'actor': {'login': 'WishMinisterMenhir42'}, 'created_at': datetime.datetime(2026, 11, 9, 7, 3, 41, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC')), 'repo': {'name': 'WishMinisterMenhir42/repo-xugazwap'}}
{'_id': '26629816406', 'actor': {'login': 'github-actions[bot]'}, 'created_at': datetime.datetime(2026, 11, 9, 7, 3, 41, tzinfo=F

In [ ]:
# eventos por tipo del ultimo dia completo; el $match usa el indice ix_created_at
from datetime import timedelta
d1 = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
d0 = d1 - timedelta(days=1)
pipeline = [{'$match': {'created_at': {'$gte': d0, '$lt': d1}}},
            {'$group': {'_id': '$type', 'n': {'$sum': 1}}}, {'$sort': {'n': -1}}, {'$limit': 8}]
print('día:', d0.date())
pprint(list(col.aggregate(pipeline)))

## U · Actualizar documento

In [7]:
res = col.update_one({'_id': demo_id}, {'$set': {'payload.action': 'stopped', 'revisado': True}, '$inc': {'veces_editado': 1}})
print('matched:', res.matched_count, '| modified:', res.modified_count)
pprint(col.find_one({'_id': demo_id}))

matched: 1 | modified: 1
{'_id': 'demo-201732',
 'actor': {'id': 1, 'login': 'estudiante_utec'},
 'created_at': datetime.datetime(2026, 11, 10, 20, 17, 32, 988000, tzinfo=FixedOffset(datetime.timedelta(0), 'UTC')),
 'payload': {'action': 'stopped'},
 'public': True,
 'repo': {'id': 1, 'name': 'utec/bigdata-demo'},
 'revisado': True,
 'type': 'WatchEvent',
 'veces_editado': 1}


## D · Eliminar documento

In [8]:
res = col.delete_one({'_id': demo_id})
print('eliminados:', res.deleted_count)
print('existe?', col.find_one({'_id': demo_id}) is not None)

eliminados: 1
existe? False
